# Contrastive Experiments Analysis

This notebook analyzes the performance of different contrastive experiments using multiple models (Qwen3-14B, Qwen3-32B). We compare different non-activating sources (random, co-occurrence, faiss, decoder_similarity) and ranking strategies (top vs quantiles) across multiple metrics.

## Analysis Overview

- **Mean Frequency-Weighted F1 Scores**: Bar charts comparing performance across configurations
- **Performance Distribution**: KDE plots showing density distributions of F1 scores
- **Comprehensive Comparison**: Side-by-side analysis of all available contrastive experiments

## Configuration Pattern

Experiments follow the pattern: `{model}_{source}_{mode}_{ranking}`
- **Sources**: random (baseline), co-occurrence, faiss, decoder_similarity
- **Modes**: contrastive_scorer_only, baseline
- **Ranking**: top, quantiles

## 1. Setup and Configuration

Import required libraries and set up the analysis environment.

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from scipy import stats
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from bootstrap_ci import compute_weighted_ci_errors

# Add the parent directory to the path to import delphi modules
sys.path.append(str(Path.cwd().parent))

from delphi.log.result_analysis import (
    import_plotly,
    load_data,
    get_agg_metrics,
    add_latent_f1,
    compute_confusion,
    compute_classification_metrics,
    frequency_weighted_f1
)

# Import plotly for plotting
px = import_plotly()

# Configuration
EXPERIMENT_DIR = "Contrastive Scoring Test"
CONFIDENCE_LEVEL = 0.95
ENABLE_BOOTSTRAP = True
BOOTSTRAP_SAMPLES = 1000
N_JOBS = 8  # Number of threads for parallel bootstrap processing

# Set up directories
results_dir = Path.cwd().parent / "results"
visualizations_dir = results_dir / "visualizations" / "contrastive_experiments"
visualizations_dir.mkdir(exist_ok=True, parents=True)

print(f"Experiment directory: {EXPERIMENT_DIR}")
print(f"Results directory: {results_dir}")
print(f"Visualizations output: {visualizations_dir}")

# Define color scheme for different experiment types
SOURCE_COLORS = {
    'random': '#E53E3E',        # Red for baseline
    'co-occurrence': '#3182CE', # Blue
    'faiss': '#38A169',         # Green  
    'decoder_similarity': '#805AD5'  # Purple
}

RANKING_STYLES = {
    'top': 'solid',
    'quantiles': 'dashed'
}

# Clean display name mappings
SOURCE_DISPLAY = {
    'random': 'Random',
    'co-occurrence': 'Co-occurrence',
    'faiss': 'FAISS',
    'decoder_similarity': 'Decoder Sim.'
}

RANKING_DISPLAY = {
    'top': 'Top',
    'quantiles': 'Quantiles'
}

def parse_experiment_name(exp_name):
    """Parse experiment directory name to extract components."""
    # Remove pythiaST_ prefix
    name = exp_name.replace('pythiaST_', '')
    
    # Extract model name (first part before the source)
    # Pattern: Qwen3_14B_quantized_w4a16 or Qwen3_32B_quantized_w4a16
    model = None
    model_display = None
    if 'Qwen3_14B' in name:
        model = 'Qwen3_14B'
        model_display = 'Qwen3-14B'
    elif 'Qwen3_32B' in name:
        model = 'Qwen3_32B'
        model_display = 'Qwen3-32B'
    else:
        # Try to extract the first part as model name
        parts = name.split('_')
        if len(parts) > 1:
            model = '_'.join(parts[:2])  # Take first two parts as model name
            model_display = model
    
    # Find the source (random, co-occurrence, faiss, decoder)
    source = None
    if 'random' in name:
        source = 'random'
    elif 'co-occurrence' in name:
        source = 'co-occurrence'
    elif 'faiss' in name:
        source = 'faiss'
    elif 'decoder' in name:
        source = 'decoder_similarity'
    
    # Find the ranking strategy
    ranking = 'quantiles' if 'quantiles' in name else 'top'
    
    # Determine mode
    mode = 'baseline' if 'baseline' in name else 'contrastive'
    
    # Create clean display name
    source_display = SOURCE_DISPLAY.get(source, source)
    ranking_display = RANKING_DISPLAY.get(ranking, ranking)
    display_name = f"{source_display} ({ranking_display})"
    
    return {
        'model': model,
        'model_display': model_display,
        'source': source,
        'ranking': ranking,
        'mode': mode,
        'display_name': display_name
    }

# Check available experiments
experiments_base = results_dir / "pythiaST" / EXPERIMENT_DIR
if experiments_base.exists():
    print(f"\nAvailable contrastive experiments:")
    model_groups = {}
    for exp_dir in sorted(experiments_base.glob('pythiaST_*')):
        config = parse_experiment_name(exp_dir.name)
        model_name = config['model_display']
        if model_name not in model_groups:
            model_groups[model_name] = []
        model_groups[model_name].append(config['display_name'])
        print(f"  - {exp_dir.name} -> {model_name}: {config['display_name']}")
    
    print(f"\nFound {len(model_groups)} model types:")
    for model_name, experiments in model_groups.items():
        print(f"  {model_name}: {len(experiments)} experiments")
else:
    print(f"Warning: Experiments directory not found: {experiments_base}")

## 2. Load Contrastive Experiment Results

Load results from all available contrastive experiments and process them for analysis.

In [ ]:
def load_contrastive_results(results_dir: Path, experiment_dir: str):
    """Load results from all contrastive experiments."""
    experiment_results = {}
    experiment_stats = {}
    
    experiments_base = results_dir / "pythiaST" / experiment_dir
    
    if not experiments_base.exists():
        print(f"Warning: Experiments directory not found: {experiments_base}")
        return experiment_results, experiment_stats
    
    for exp_dir in experiments_base.glob('pythiaST_*'):
        config = parse_experiment_name(exp_dir.name)
        # Create unique key that includes model name
        unique_key = f"{config['model_display']}: {config['display_name']}"
        
        scores_path = exp_dir / "scores"
        if scores_path.exists():
            try:
                # Use the shared cache from the parent pythiaST directory
                latents_path = results_dir / "pythiaST" / "cache" / "latents"
                if not latents_path.exists():
                    latents_path = exp_dir / "latents"  # Fallback to local latents
                
                if latents_path.exists():
                    # Extract module names from the actual files
                    sample_score_dir = next(scores_path.iterdir())
                    sample_files = list(sample_score_dir.glob("*.txt"))
                    if sample_files:
                        # Extract module name from filename pattern
                        sample_filename = sample_files[0].stem
                        module_name = sample_filename.split('_latent')[0]
                        modules = [module_name]
                    else:
                        print(f"No score files found in {sample_score_dir}")
                        continue
                    
                    # Use load_data from result_analysis.py
                    latent_df, counts = load_data(scores_path, latents_path, modules)
                    
                    if latent_df.empty:
                        print(f"No data found for {unique_key}")
                        continue
                    
                    # Use add_latent_f1 and get_agg_metrics from result_analysis.py
                    latent_df = add_latent_f1(latent_df)
                    processed_df = get_agg_metrics(latent_df, counts)
                    
                    experiment_results[unique_key] = {
                        'latent_df': latent_df,
                        'processed_df': processed_df,
                        'counts': counts,
                        'config': config
                    }
                else:
                    print(f"Latents path not found for {display_name}")
            
            except Exception as e:
                print(f"Error loading results for {exp_dir.name}: {e}")
                continue
        
        # Load experiment statistics
        stats_file = exp_dir / "explainer_stats.json"
        if stats_file.exists():
            try:
                with open(stats_file, 'r') as f:
                    stats = json.load(f)
                    experiment_stats[unique_key] = stats
            except Exception as e:
                print(f"Error loading stats for {exp_dir.name}: {e}")
                experiment_stats[unique_key] = None
        else:
            experiment_stats[unique_key] = None
    
    return experiment_results, experiment_stats

# Load all contrastive experiment results
print("Loading contrastive experiment results...")
experiment_results, experiment_stats = load_contrastive_results(results_dir, EXPERIMENT_DIR)

print(f"\nLoaded results for {len(experiment_results)} experiments:")
for exp_name in experiment_results.keys():
    print(f"  - {exp_name}")

# Display sample metrics for the first experiment
if experiment_results:
    sample_exp = list(experiment_results.keys())[0]
    sample_data = experiment_results[sample_exp]['processed_df']
    print(f"\nSample metrics from {sample_exp}:")
    print(sample_data[['score_type', 'accuracy', 'f1_score', 'precision', 'recall', 'weighted_f1']].round(3))

## 3. Generate Mean Frequency-Weighted F1 Bar Charts

Create bar charts showing mean frequency-weighted F1 scores across all contrastive experiments, organized by source and ranking strategy.

In [ ]:
# Bootstrap CI computation is now imported from bootstrap_ci module

def compute_contrastive_error_bars(experiment_results, score_type, enable_bootstrap=ENABLE_BOOTSTRAP, n_boot=BOOTSTRAP_SAMPLES, confidence=CONFIDENCE_LEVEL, n_jobs=N_JOBS):
    """Compute frequency-weighted F1 and CI errors for contrastive experiments."""
    rows = []
    for exp_name, data in experiment_results.items():
        processed_df = data['processed_df']
        score_row = processed_df[processed_df['score_type'] == score_type]
        if len(score_row) == 0:
            continue
        freq_weighted_f1 = score_row['weighted_f1'].iloc[0] if 'weighted_f1' in score_row.columns else None
        latent_df = data['latent_df']
        counts = data['counts']
        config = data['config']
        score_subset = latent_df[latent_df['score_type'] == score_type]
        
        if len(score_subset) == 0 or counts is None or freq_weighted_f1 is None:
            rows.append({
                'experiment': config['display_name'],  # Use clean display name
                'model': config['model_display'],
                'source': config['source'],
                'ranking': config['ranking'],
                'mode': config['mode'],
                'frequency_weighted_f1': freq_weighted_f1,
                'ci_lower_error': 0.0,
                'ci_upper_error': 0.0
            })
            continue
        
        if enable_bootstrap:
            # Use bootstrap resampling within latents (with multiprocessing)
            ci_lower_error, ci_upper_error = compute_weighted_ci_errors(score_subset, counts, freq_weighted_f1, confidence, n_boot, n_jobs)
        else:
            ci_lower_error = 0.0
            ci_upper_error = 0.0
        
        rows.append({
            'experiment': config['display_name'],  # Use clean display name
            'model': config['model_display'],
            'source': config['source'],
            'ranking': config['ranking'],
            'mode': config['mode'],
            'frequency_weighted_f1': float(freq_weighted_f1),
            'ci_lower_error': ci_lower_error,
            'ci_upper_error': ci_upper_error
        })
    return pd.DataFrame(rows)

# Generate bar charts for each score type
all_score_types = set()
for data in experiment_results.values():
    all_score_types.update(list(data['processed_df']['score_type'].unique()))
all_score_types = sorted(list(all_score_types))

print(f"Generating bar charts for score types: {all_score_types}")

# Compute error bars for all score types
contrastive_error_tables = {}
for st in all_score_types:
    print(f"Computing error bars for {st} (bootstrap={ENABLE_BOOTSTRAP})")
    contrastive_error_tables[st] = compute_contrastive_error_bars(experiment_results, st)

def score_display_name(st):
    return 'Fuzzing' if str(st).lower() == 'fuzz' else str(st).title()

# Create bar charts - separate plots for each model
for score_type, error_df in contrastive_error_tables.items():
    if error_df.empty:
        continue
    
    # Get unique models
    models = error_df['model'].unique()
    
    # Create a separate plot for each model
    for model_name in sorted(models):
        model_df = error_df[error_df['model'] == model_name].copy()
        
        if model_df.empty:
            continue
        
        # Calculate random baseline F1 from class distribution
        # Get the first experiment result for this model to compute baseline
        model_experiments = [k for k, v in experiment_results.items() if v['config']['model_display'] == model_name]
        if model_experiments:
            first_exp = experiment_results[model_experiments[0]]
            score_subset = first_exp['latent_df'][first_exp['latent_df']['score_type'] == score_type]
            if len(score_subset) > 0:
                # Calculate overall positive rate across all examples
                total_positives = score_subset['activating'].sum()
                total_examples = len(score_subset)
                p = total_positives / total_examples if total_examples > 0 else 0.5
                
                # For a random classifier predicting positive with probability p:
                # Precision = p (ratio of true positives among predicted positives)
                # Recall = p (ratio of predicted positives among true positives)
                # F1 = 2 * p * p / (p + p) = p
                random_baseline_f1 = p
            else:
                random_baseline_f1 = 0.5
        else:
            random_baseline_f1 = 0.5
        
        # Sort by frequency_weighted_f1 for better visualization
        model_df = model_df.sort_values('frequency_weighted_f1', ascending=False)
        
        # Create colors based on source
        colors = [SOURCE_COLORS.get(source, '#808080') for source in model_df['source']]
        
        fig = px.bar(
            model_df,
            x='experiment',
            y='frequency_weighted_f1',
            color='source',
            color_discrete_map=SOURCE_COLORS,
            title=f'{model_name} - Frequency-Weighted F1 Score - {score_display_name(score_type)} ({int(CONFIDENCE_LEVEL*100)}% CI)',
            text='frequency_weighted_f1',
            hover_data=['ranking', 'mode']
        )
        
        # Add error bars
        fig.update_traces(
            error_y=dict(
                type='data',
                symmetric=False,
                array=model_df['ci_upper_error'],
                arrayminus=model_df['ci_lower_error']
            )
        )
        
        # Add random baseline as a red dotted horizontal line
        fig.add_hline(
            y=random_baseline_f1,
            line_dash="dot",
            line_color="red",
            line_width=2,
            annotation_text=f"Random Baseline (F1={random_baseline_f1:.3f})",
            annotation_position="right"
        )
        
        # Adjust y-axis range to ensure baseline is visible
        y_min = min(0, random_baseline_f1 - 0.1)
        y_max = max(1, model_df['frequency_weighted_f1'].max() + 0.1)
        
        fig.update_layout(
            yaxis_range=[y_min, y_max],
            xaxis_title="Experiment Configuration",
            yaxis_title=f"Frequency-Weighted F1 Score ({int(CONFIDENCE_LEVEL*100)}% CI)",
            xaxis={'tickangle': 45},
            height=600,
            legend_title="Non-Activating Source"
        )
        
        fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
        fig.show()
        
        # Save the chart with model name in filename
        model_safe = model_name.replace('-', '').replace(' ', '_')
        output_file_pdf = visualizations_dir / f"contrastive_freq_weighted_f1_bar_{model_safe}_{score_type}.pdf"
        output_file_png = visualizations_dir / f"contrastive_freq_weighted_f1_bar_{model_safe}_{score_type}.png"
        fig.write_image(str(output_file_pdf))
        fig.write_image(str(output_file_png))
        print(f"Saved contrastive F1 bar chart for {model_name}: {output_file_pdf}")

print(f"\nBar charts saved to {visualizations_dir}")

### 3.1 Model Comparison Charts

Create side-by-side comparison charts to compare the same configurations across different models.

In [ ]:
# Create comparison charts showing the same configuration across different models
print("\nGenerating model comparison charts...")

for score_type, error_df in contrastive_error_tables.items():
    if error_df.empty:
        continue
    
    # Create a grouped bar chart comparing models for each source+ranking combination
    # Pivot the data to have models as separate bars
    comparison_df = error_df.copy()
    comparison_df['config_key'] = comparison_df['source'] + ' (' + comparison_df['ranking'] + ')'
    
    fig = px.bar(
        comparison_df,
        x='config_key',
        y='frequency_weighted_f1',
        color='model',
        barmode='group',
        title=f'Model Comparison - Frequency-Weighted F1 Score - {score_display_name(score_type)} ({int(CONFIDENCE_LEVEL*100)}% CI)',
        text='frequency_weighted_f1',
        hover_data=['experiment', 'mode']
    )
    
    # Add error bars
    fig.update_traces(
        error_y=dict(
            type='data',
            symmetric=False,
            array=comparison_df['ci_upper_error'],
            arrayminus=comparison_df['ci_lower_error']
        )
    )
    
    fig.update_layout(
        yaxis_range=[0, 1],
        xaxis_title="Configuration (Source + Ranking)",
        yaxis_title=f"Frequency-Weighted F1 Score ({int(CONFIDENCE_LEVEL*100)}% CI)",
        xaxis={'tickangle': 45},
        height=600,
        legend_title="Model"
    )
    
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig.show()
    
    # Save the comparison chart
    output_file_pdf = visualizations_dir / f"contrastive_model_comparison_{score_type}.pdf"
    output_file_png = visualizations_dir / f"contrastive_model_comparison_{score_type}.png"
    fig.write_image(str(output_file_pdf))
    fig.write_image(str(output_file_png))
    print(f"Saved model comparison chart: {output_file_pdf}")

print(f"\nModel comparison charts saved to {visualizations_dir}")

## 4. Generate KDE Distribution Plots

Create kernel density estimation plots showing the distribution of F1 scores across different experimental configurations.

In [ ]:
def prepare_kde_data(experiment_results, score_type):
    """Prepare data for KDE plots by extracting per-latent F1 scores."""
    kde_data = []
    
    for exp_name, data in experiment_results.items():
        latent_df = data['latent_df']
        counts = data['counts']
        config = data['config']
        
        # Filter for the specific score type
        score_subset = latent_df[latent_df['score_type'] == score_type]
        
        if len(score_subset) == 0 or counts is None:
            continue
        
        # Extract per-latent F1 scores
        for (module, latent_idx), grp in score_subset.groupby(["module", "latent_idx"]):
            if module in counts and latent_idx < len(counts[module]):
                f1 = compute_classification_metrics(compute_confusion(grp))["f1_score"]
                firing_count = counts[module][latent_idx].item()
                
                kde_data.append({
                    'experiment': exp_name,
                    'source': config['source'],
                    'ranking': config['ranking'],
                    'mode': config['mode'],
                    'f1_score': float(f1),
                    'firing_count': float(firing_count),
                    'module': module,
                    'latent_idx': latent_idx
                })
    
    return pd.DataFrame(kde_data)

# Set up matplotlib style for KDE plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Generate separate KDE plots for each ranking strategy
for score_type in all_score_types:
    print(f"Generating KDE plots for {score_type}...")
    
    kde_data = prepare_kde_data(experiment_results, score_type)
    
    if kde_data.empty:
        print(f"No data available for {score_type} KDE plot")
        continue
    
    # Create separate plots for each ranking strategy
    ranking_strategies = ['quantiles', 'top']
    
    for ranking_strategy in ranking_strategies:
        ranking_data = kde_data[kde_data['ranking'] == ranking_strategy]
        
        if ranking_data.empty:
            print(f"No data for {ranking_strategy} strategy in {score_type}")
            continue
        
        # Create figure comparing all sources for this ranking strategy
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        
        sources = sorted(ranking_data['source'].unique())
        
        for source in sources:
            source_data = ranking_data[ranking_data['source'] == source]
            
            if len(source_data) > 0:
                # Weight by firing count for the KDE
                weights = source_data['firing_count'].values
                weights = weights / weights.sum()  # Normalize weights
                
                # Create KDE plot with clean labels
                source_label = SOURCE_DISPLAY.get(source, source)
                sns.kdeplot(
                    data=source_data,
                    x='f1_score',
                    weights=weights,
                    ax=ax,
                    label=source_label,
                    color=SOURCE_COLORS.get(source, '#808080'),
                    linewidth=3,
                    alpha=0.8
                )
        
        # Styling
        ranking_display = RANKING_DISPLAY.get(ranking_strategy, ranking_strategy)
        ax.set_title(f'{score_display_name(score_type)} F1 Score Distribution - {ranking_display} Strategy', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('F1 Score', fontsize=14, fontweight='medium')
        ax.set_ylabel('Density', fontsize=14, fontweight='medium')
        ax.legend(title='Non-Activating Source', title_fontsize=12, fontsize=11, frameon=True, 
                 fancybox=True, shadow=True, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 1)
        
        # Add subtle background styling
        ax.set_facecolor('#FAFAFA')
        
        plt.tight_layout()
        
        # Show the plot
        plt.show()
        
        # Save the plot
        output_file_pdf = visualizations_dir / f"contrastive_kde_{score_type}_{ranking_strategy}.pdf"
        output_file_png = visualizations_dir / f"contrastive_kde_{score_type}_{ranking_strategy}.png"
        plt.savefig(str(output_file_pdf), dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(str(output_file_png), dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved KDE plot: {output_file_pdf}")
        
        plt.close()

print(f"\nKDE plots saved to {visualizations_dir}")

## 6. Random vs Co-occurrence Performance Comparison

Scatter plot analysis to see if the performance drop from random to co-occurrence non-activating sources is consistent across latents, or if some latents drop more than others.

In [ ]:
def prepare_random_vs_cooccurrence_comparison(experiment_results, score_type):
    """Prepare data comparing random vs co-occurrence performance per latent."""
    comparison_data = []
    
    # Group experiments by model and ranking to find matching pairs
    for model_name in set(v['config']['model_display'] for v in experiment_results.values()):
        for ranking_strategy in ['top', 'quantiles']:
            # Find random and co-occurrence experiments for this model/ranking combo
            random_exp = None
            cooccurrence_exp = None
            
            for exp_name, data in experiment_results.items():
                config = data['config']
                if (config['model_display'] == model_name and 
                    config['ranking'] == ranking_strategy):
                    if config['source'] == 'random':
                        random_exp = exp_name
                    elif config['source'] == 'co-occurrence':
                        cooccurrence_exp = exp_name
            
            # If we have both, compare them
            if random_exp and cooccurrence_exp:
                random_data = experiment_results[random_exp]
                cooccur_data = experiment_results[cooccurrence_exp]
                
                random_latent_df = random_data['latent_df']
                cooccur_latent_df = cooccur_data['latent_df']
                counts = random_data['counts']
                
                # Filter for the specific score type
                random_subset = random_latent_df[random_latent_df['score_type'] == score_type]
                cooccur_subset = cooccur_latent_df[cooccur_latent_df['score_type'] == score_type]
                
                if len(random_subset) == 0 or len(cooccur_subset) == 0 or counts is None:
                    continue
                
                # Extract per-latent F1 scores
                random_f1_map = {}
                cooccur_f1_map = {}
                
                for (module, latent_idx), grp in random_subset.groupby(["module", "latent_idx"]):
                    if module in counts and latent_idx < len(counts[module]):
                        f1 = compute_classification_metrics(compute_confusion(grp))["f1_score"]
                        firing_count = counts[module][latent_idx].item()
                        random_f1_map[(module, latent_idx)] = {
                            'f1': float(f1),
                            'firing_count': float(firing_count)
                        }
                
                for (module, latent_idx), grp in cooccur_subset.groupby(["module", "latent_idx"]):
                    if module in counts and latent_idx < len(counts[module]):
                        f1 = compute_classification_metrics(compute_confusion(grp))["f1_score"]
                        cooccur_f1_map[(module, latent_idx)] = float(f1)
                
                # Match latents that appear in both
                for (module, latent_idx), random_info in random_f1_map.items():
                    if (module, latent_idx) in cooccur_f1_map:
                        comparison_data.append({
                            'model': model_name,
                            'ranking': ranking_strategy,
                            'module': module,
                            'latent_idx': latent_idx,
                            'random_f1': random_info['f1'],
                            'cooccurrence_f1': cooccur_f1_map[(module, latent_idx)],
                            'f1_drop': random_info['f1'] - cooccur_f1_map[(module, latent_idx)],
                            'firing_count': random_info['firing_count']
                        })
    
    return pd.DataFrame(comparison_data)

# Generate scatter plots for each score type
for score_type in all_score_types:
    print(f"Generating random vs co-occurrence comparison for {score_type}...")
    
    comparison_df = prepare_random_vs_cooccurrence_comparison(experiment_results, score_type)
    
    if comparison_df.empty:
        print(f"No comparison data available for {score_type}")
        continue
    
    # Add log-scaled firing count for better color mapping
    comparison_df['log_firing_count'] = np.log10(comparison_df['firing_count'] + 1)
    
    # Create separate plots for each model and ranking strategy
    models = comparison_df['model'].unique()
    ranking_strategies = comparison_df['ranking'].unique()
    
    for model_name in sorted(models):
        for ranking_strategy in sorted(ranking_strategies):
            subset_df = comparison_df[
                (comparison_df['model'] == model_name) & 
                (comparison_df['ranking'] == ranking_strategy)
            ]
            
            if subset_df.empty:
                continue
            
            # Create scatter plot
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            
            # Scatter plot with color by firing frequency
            scatter = ax.scatter(
                subset_df['random_f1'],
                subset_df['cooccurrence_f1'],
                c=subset_df['log_firing_count'],
                cmap='viridis',
                s=50,
                alpha=0.6,
                edgecolors='black',
                linewidth=0.5
            )
            
            # Add diagonal line (where random = co-occurrence)
            min_val = 0
            max_val = 1
            ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=2, label='Equal Performance')
            
            # Add colorbar
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label('Log10(Firing Count + 1)', fontsize=12)
            
            # Calculate statistics
            mean_drop = subset_df['f1_drop'].mean()
            median_drop = subset_df['f1_drop'].median()
            points_below_diagonal = (subset_df['cooccurrence_f1'] < subset_df['random_f1']).sum()
            total_points = len(subset_df)
            pct_below = (points_below_diagonal / total_points * 100) if total_points > 0 else 0
            
            # Add statistics text box
            stats_text = (
                f'Mean drop: {mean_drop:.3f}\n'
                f'Median drop: {median_drop:.3f}\n'
                f'Points below diagonal: {points_below_diagonal}/{total_points} ({pct_below:.1f}%)'
            )
            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
                   fontsize=11, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
            
            # Styling
            ranking_display = RANKING_DISPLAY.get(ranking_strategy, ranking_strategy)
            ax.set_title(
                f'{model_name} - {score_display_name(score_type)}\n'
                f'Random vs Co-occurrence F1 Scores ({ranking_display} Strategy)',
                fontsize=14, fontweight='bold', pad=15
            )
            ax.set_xlabel('Random Non-Activating F1 Score', fontsize=12, fontweight='medium')
            ax.set_ylabel('Co-occurrence Non-Activating F1 Score', fontsize=12, fontweight='medium')
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_aspect('equal')
            
            plt.tight_layout()
            plt.show()
            
            # Save the plot
            model_safe = model_name.replace('-', '').replace(' ', '_')
            output_file_pdf = visualizations_dir / f"random_vs_cooccurrence_{model_safe}_{ranking_strategy}_{score_type}.pdf"
            output_file_png = visualizations_dir / f"random_vs_cooccurrence_{model_safe}_{ranking_strategy}_{score_type}.png"
            plt.savefig(str(output_file_pdf), dpi=300, bbox_inches='tight')
            plt.savefig(str(output_file_png), dpi=300, bbox_inches='tight')
            print(f"Saved scatter plot: {output_file_pdf}")
            
            plt.close()

print(f"\nScatter plots saved to {visualizations_dir}")